# Step 18 — level 1, alone: each site finds its own endotypes

**Data type: RNA_array** (GSE65391). **Reads:** `step10_site_{A,B,C}.rds`, `step04_genes.rds`.
**Writes:** `step18_site_{A,B,C}.rds`.

This is the baseline for measuring what federation buys, as in `endotypes-proteomics` step 19. Each
site runs the endotype recipe on its own discovery patients (about 37) with **no exchange at all**:

- its own 1,000 most variable genes;
- its own centring and its own principal components;
- its own consensus clustering and its own choice of k;
- its own centroids.

The data are the corrected data from step 06, so the comparison isolates the clustering, not the
batch correction. Round 1's central functions (`consensus_matrix`, `labels_from_consensus` in
`src/endotypes.R`) run here unchanged, each on one site's patients.

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
genes <- readRDS(art("step04_genes.rds")); expr <- genes$keep[genes$expressed]
fit_site <- function(E, core, gene_list = NULL, seed) {
  set.seed(seed)
  X <- E[expr, core]
  top <- if (is.null(gene_list)) names(sort(apply(X, 1, var), decreasing = TRUE))[1:1000] else gene_list
  centre <- rowMeans(X[top, ])
  Z  <- t(X[top, ] - centre)
  S  <- prcomp(Z)$x[, 1:10]
  C  <- lapply(setNames(2:6, 2:6), function(k) consensus_matrix(S, k, n_resamples = 100))
  pt <- data.frame(k = 2:6, pac = sapply(C, pac))
  k  <- choose_k(pt)
  lab <- labels_from_consensus(C[[as.character(k)]], k)
  centroids <- t(sapply(levels(lab), function(e) colMeans(Z[lab == e, , drop = FALSE])))
  list(genes = top, centre = centre, centroids = centroids, k = k, pac = pt, labels = lab)
}
core_of <- function(m) { d <- m[m$split == "discovery", ]; d <- d[order(d$subject, d$visit), ]
                         rownames(d)[!duplicated(d$subject)] }

In [2]:
for (i in seq_along(SITES)) {
  s <- SITES[i]; d <- readRDS(site_file("10", s))
  model <- fit_site(d$E, core_of(d$meta), seed = SEED + 180L + i)
  saveRDS(model, site_file("18", s))
  cat(sprintf("site %s alone: k = %d, PAC %.3f, sizes %s\n", s, model$k,
              model$pac$pac[model$pac$k == model$k], paste(table(model$labels), collapse = "/")))
}

site A alone: k = 6, PAC 0.338, sizes 10/9/8/5/4/1
site B alone: k = 5, PAC 0.245, sizes 11/10/9/5/2
site C alone: k = 2, PAC 0.057, sizes 20/16


In [3]:
sapply(SITES, function(s) readRDS(site_file("18", s))$pac$pac)

A,B,C
0.4189189,0.4324324,0.05714286
0.4009009,0.5435435,0.44444444
0.3678679,0.3558559,0.33333333
0.3603604,0.2447447,0.32857143
0.3378378,0.2552553,0.30952381


## How much do the sites' gene lists overlap?

In [4]:
g <- lapply(setNames(SITES, SITES), function(s) readRDS(site_file("18", s))$genes)
c(A_and_B = length(intersect(g$A, g$B)), A_and_C = length(intersect(g$A, g$C)),
  B_and_C = length(intersect(g$B, g$C)), all_three = length(Reduce(intersect, g)))

A_and_B   A_and_C   B_and_C all_three 
      757       719       715       634

## Findings

**Alone, the three sites do not agree on how many endotypes there are.**
- **Site C** finds two clean groups (k = 2, PAC 0.06).
- **Site A** finds no stable two-group split (PAC 0.42 at k = 2). It takes k = 6, whose smallest
  cluster is one patient.
- **Site B** takes k = 5 (PAC 0.24), with clusters of 2 and 5 patients.

With about 37 patients per site, the consensus is noisy. The same biology that gives a clean k = 2 when
the sites work together (step 12, pooled PAC 0.155) is not visible at two of the three sites alone.
The sites' own gene lists also differ: only 634 of each site's 1,000 genes are common to all three.